In [2]:
!pip install flask flask-cors pyngrok transformers torch -q

In [3]:
from pyngrok import ngrok
ngrok.set_auth_token("3E1017fgW6WMaOdrXlZqQYCgorp_6nYApQt5xWHeT4drojRW5")  # Get free token at dashboard.ngrok.com

In [5]:
from flask import Flask, request, jsonify
from flask_cors import CORS
from transformers import pipeline
from pyngrok import ngrok
import threading

app = Flask(__name__)
CORS(app)

# ✅ Use sentiment-analysis instead of summarization
print("Loading AI model...")
classifier = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")
print("Model ready!")

@app.route("/analyze", methods=["POST"])
def analyze():
    data = request.json
    text = data.get("text", "")
    if not text or len(text) < 5:
        return jsonify({"error": "Text too short."}), 400

    result = classifier(text[:512])[0]   # distilbert max = 512 tokens
    return jsonify({
        "label": result["label"],             # POSITIVE / NEGATIVE
        "score": round(result["score"] * 100, 1)   # confidence %
    })

@app.route("/health", methods=["GET"])
def health():
    return jsonify({"status": "running", "model": "distilbert-sst2"})

def run_flask():
    app.run(port=5000)

thread = threading.Thread(target=run_flask)
thread.daemon = True
thread.start()

public_url = ngrok.connect(5000)
print(f"\n✅ Live at: {public_url}")

Loading AI model...


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Model ready!
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit



✅ Live at: NgrokTunnel: "https://penniless-outpost-emblaze.ngrok-free.dev" -> "http://localhost:5000"


In [8]:
import requests

API_URL = str("https://penniless-outpost-emblaze.ngrok-free.dev")

tests = [
    "This product is absolutely amazing, I love it!",
    "Terrible experience, completely disappointed.",
    "It was okay, nothing special."
]

for t in tests:
    r = requests.post(f"{API_URL}/analyze", json={"text": t})
    d = r.json()
    print(f"{d['label']} ({d['score']}%) — {t[:50]}")

INFO:werkzeug:127.0.0.1 - - [21/May/2026 02:34:28] "POST /analyze HTTP/1.1" 200 -


POSITIVE (100.0%) — This product is absolutely amazing, I love it!


INFO:werkzeug:127.0.0.1 - - [21/May/2026 02:34:28] "POST /analyze HTTP/1.1" 200 -


NEGATIVE (100.0%) — Terrible experience, completely disappointed.


INFO:werkzeug:127.0.0.1 - - [21/May/2026 02:34:28] "POST /analyze HTTP/1.1" 200 -


NEGATIVE (98.2%) — It was okay, nothing special.
